# Loading Data

In [120]:
import pandas as pd
import numpy as np

In [121]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submit = pd.read_csv("sample_submission.csv")

In [122]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


# Logistic Regression cannot work with missing value, thus I need to impute the original data

In [123]:
# You must explicitly enable IterativeImputer as it is still experimental
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

In [124]:
# Load numeric data set
def iterative_imputer(df):
    string_cols = ["gender", "stress_level","academic_work_impact"]

    print(df.columns)

    df = df.drop(columns=string_cols)
    # 1. Initialize the imputer
    # sample_posterior=True adds random noise back, protecting the distribution shape
    imputer = IterativeImputer(sample_posterior=True, max_iter=10, random_state=42)

    # 2. Fit and transform the data (returns a numpy array)
    df_array = imputer.fit_transform(df)

    # 3. Convert back to a pandas DataFrame
    df_imputed = pd.DataFrame(df_array, columns=df.columns)

    print(df.columns)

    return df_imputed

In [125]:
train_imputed = iterative_imputer(train)

Index(['id', 'age', 'daily_screen_time_hours', 'social_media_hours',
       'gaming_hours', 'work_study_hours', 'sleep_hours',
       'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
       'gender', 'stress_level', 'academic_work_impact', 'addicted_label'],
      dtype='str')
Index(['id', 'age', 'daily_screen_time_hours', 'social_media_hours',
       'gaming_hours', 'work_study_hours', 'sleep_hours',
       'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
       'addicted_label'],
      dtype='str')


In [126]:
train_imputed.dtypes

id                         float64
age                        float64
daily_screen_time_hours    float64
social_media_hours         float64
gaming_hours               float64
work_study_hours           float64
sleep_hours                float64
notifications_per_day      float64
app_opens_per_day          float64
weekend_screen_time        float64
addicted_label             float64
dtype: object

In [127]:
test_imputed = iterative_imputer(test)

Index(['id', 'age', 'daily_screen_time_hours', 'social_media_hours',
       'gaming_hours', 'work_study_hours', 'sleep_hours',
       'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',
       'gender', 'stress_level', 'academic_work_impact'],
      dtype='str')
Index(['id', 'age', 'daily_screen_time_hours', 'social_media_hours',
       'gaming_hours', 'work_study_hours', 'sleep_hours',
       'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time'],
      dtype='str')


In [128]:
train_missing = pd.DataFrame(train_imputed.isna().sum()).reset_index()
train_missing = train_missing.rename(columns={"index":"columnName",0:"countMissing"})
train_missing['percent_missing']= (train_missing['countMissing']/train.shape[0])*100
train_missing

,columnName,countMissing,percent_missing
0,id,0,0.0
1,age,0,0.0
2,daily_screen_time_hours,0,0.0
3,social_media_hours,0,0.0
4,gaming_hours,0,0.0
5,work_study_hours,0,0.0
6,sleep_hours,0,0.0
7,notifications_per_day,0,0.0
8,app_opens_per_day,0,0.0
9,weekend_screen_time,0,0.0


# Add Feature

In [129]:
# 2. Identify your string columns
string_cols = ["gender", "stress_level","academic_work_impact"]

# 3. Convert string columns to pandas 'category' type
for col in string_cols:
    train_imputed[col] = train[col].astype("category")
    test_imputed[col] = test[col].astype("category")

In [130]:
# adding feature
# logictic regression cannot work with categorical variable, thus change categorical to dummy variable 1 and 0
def add_features(df):
    df = df.copy()
    df['screen_minus_sleep'] = df['daily_screen_time_hours'] - df['sleep_hours']
    df['social_ratio'] = df['social_media_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    df['gaming_ratio'] = df['gaming_hours'] / (df['daily_screen_time_hours'] + 1e-3)
    # df['weekend_vs_daily'] = df['weekend_screen_time'] / (df['daily_screen_time_hours'] + 1e-3)
    # df['opens_per_notification'] = df['app_opens_per_day'] / (df['notifications_per_day'] + 1)
    df['leisure_hours'] = np.log((np.abs(df['social_media_hours'].fillna(0)) + np.abs(df['gaming_hours'].fillna(0)))+1)
    # df['n_missing'] = df.isna().sum(axis=1)
    df["social_media_hours_exp"] = np.abs(df['social_media_hours'])**0.3
    df["gaming_hours_exp"] = np.abs(df['gaming_hours'])**0.2
    df["work_study_hours_exp_015"] = np.abs(df['work_study_hours'])**0.15
    df["age*daily_screen_time"] = np.log((df["age"]/(np.abs(df["daily_screen_time_hours"])+1e-3))+1)
    df["age*social_media"] = np.log((df["age"]/(np.abs(df["social_media_hours"])+1e-3))+1)
    df["age*work_study"] = np.log((df["age"]/(np.abs(df["work_study_hours"])+1e-3))+1)
    # df["notif*open"] = df['app_opens_per_day'] * (df['notifications_per_day'])
    # df["notif_per_open"] = df['notifications_per_day'] / (df['app_opens_per_day'] + 1)
    # df["age*sleep_hours"] = df["age"]*df["sleep_hours"]
    df["social_vs_gaming"] = np.log((np.abs(df["social_media_hours"])/(np.abs(df["gaming_hours"])+1e-3))+1)
    df["social_vs_sleeping"] = np.abs(df["social_media_hours"]/(df["sleep_hours"]+1e-3))**0.3
    df["gaming_vs_sleeping"] = np.abs(df["gaming_hours"]/(df["sleep_hours"]+1e-3))**0.3
    df["leisure_vs_sleeping"] = (np.log((np.abs(df["leisure_hours"])/(np.abs(df["sleep_hours"])+1e-3)+1)))

    # adding dummy variable
    df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)
    df = pd.get_dummies(df, columns=['stress_level'], drop_first=True, dtype=int)
    df = pd.get_dummies(df, columns=['academic_work_impact'], drop_first=True, dtype=int)

    # drop_cols = ["age","sleep_hours","notifications_per_day","app_opens_per_day"]
    # df = df.drop(columns=drop_cols)

    return df

In [131]:
# work study hours
train_imputed = add_features(train_imputed)
test_imputed = add_features(test_imputed)

## Checking NaN Column

In [132]:
train_missing = pd.DataFrame(train_imputed.isna().sum()).reset_index()
train_missing = train_missing.rename(columns={"index":"columnName",0:"countMissing"})
train_missing['percent_missing']= (train_missing['countMissing']/train.shape[0])*100
train_missing

,columnName,countMissing,percent_missing
0,id,0,0.0
1,age,0,0.0
2,daily_screen_time_hours,0,0.0
3,social_media_hours,0,0.0
4,gaming_hours,0,0.0
5,work_study_hours,0,0.0
6,sleep_hours,0,0.0
7,notifications_per_day,0,0.0
8,app_opens_per_day,0,0.0
9,weekend_screen_time,0,0.0


In [133]:
# train_imputed[train_imputed['leisure_hours'].isna()][['social_media_hours','gaming_hours','leisure_hours']]

In [134]:
train_imputed[train_imputed['age*daily_screen_time'].isna()][['age','daily_screen_time_hours','age*daily_screen_time']]

,age,daily_screen_time_hours,age*daily_screen_time


# Modeling Preparation

In [135]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [136]:
'''
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
'''

'\nimport lightgbm as lgb\nimport xgboost as xgb\nfrom catboost import CatBoostClassifier, Pool\n'

In [137]:
train_imputed.dtypes

id                          float64
age                         float64
daily_screen_time_hours     float64
social_media_hours          float64
gaming_hours                float64
work_study_hours            float64
sleep_hours                 float64
notifications_per_day       float64
app_opens_per_day           float64
weekend_screen_time         float64
addicted_label              float64
screen_minus_sleep          float64
social_ratio                float64
gaming_ratio                float64
leisure_hours               float64
social_media_hours_exp      float64
gaming_hours_exp            float64
work_study_hours_exp_015    float64
age*daily_screen_time       float64
age*social_media            float64
age*work_study              float64
social_vs_gaming            float64
social_vs_sleeping          float64
gaming_vs_sleeping          float64
leisure_vs_sleeping         float64
gender_Male                   int64
gender_Other                  int64
stress_level_Low            

In [138]:
# 4. Separate features and target
X = train_imputed.drop(columns=["id","addicted_label"])
y = train_imputed["addicted_label"]
X_test = test_imputed.drop(columns=["id"])

In [139]:
SEED = 42
N_FOLDS = 5
TARGET = 'addicted_label'

In [140]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

# Modeling

## Logistic Regression Model

In [141]:
from sklearn.linear_model import LogisticRegression

In [142]:
def fit_logit(X_tr, y_tr, X_va, y_va):
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [144]:
oof_logit, test_logit, auc_logit = run_cv(fit_logit, 'LogitModel')

C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  fold 0: AUC = 0.92497


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  fold 1: AUC = 0.92543


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  fold 2: AUC = 0.92582


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  fold 3: AUC = 0.92673


C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  fold 4: AUC = 0.92576
LogitModel | OOF AUC = 0.92573 (mean 0.92574 ± 0.00058)


## Ensemble

In [150]:
# test_mean = (test_lgb + test_xgb + test_cat) / 3
# test_mean = (test_lgb + test_xgb ) / 2
# test_rank = (rankdata(test_lgb) + rankdata(test_xgb) + rankdata(test_cat)) / (3 * len(test_lgb))

def save_sub(pred, name):
    sub = sample_submit.copy()
    sub[TARGET] = pred
    sub.to_csv(name, index=False)
    print('guardado:', name)

# save_sub(test_lgb,  'sub_lgb.csv')
save_sub(test_logit,  'sub_lgb_12.csv')
#save_sub(test_cat,  'sub_cat.csv')
# save_sub(test_mean, 'sub_ensemble_mean.csv')
# save_sub(test_rank, 'sub_ensemble_rank.csv')

guardado: sub_lgb_12.csv


In [151]:
oof_logit

array([0.74246254, 0.36781296, 0.74304225, ..., 0.80659813, 0.98879309,
       0.39140825], shape=(691369,))

In [152]:
def save_oof(pred, name):
    oof = train.copy()
    oof = oof[["id",TARGET]]
    oof[TARGET] = pred
    oof.to_csv(name, index=False)
    print('saving:', name)

In [153]:
# save_oof(oof_lgb,'oof_lgb.csv')
save_oof(oof_logit,'oof_logit_12.csv')

saving: oof_logit_12.csv
